# Kapuã QRNG — uso da API com token pessoal (Jupyter / Python)

Este notebook usa o **contrato real da produção** (verificado em 2026-08-29 contra
`https://bongo.dobslit.com/qrng/v1` — frontend `qrng-web:9e36a90`, API `qrng-client-api:4137bfe`).
Nenhum endpoint inventado.

**O que o token faz:** autentica o acesso e contabiliza a cota diária.
**O que o token NÃO faz:** não altera, não condiciona e não melhora os bytes aleatórios.
Os bytes são os mesmos, com ou sem token; o token só muda o limite por requisição
(1 MiB autenticado vs 64 KiB anônimo) e a cota.

**Proveniência:** a produção informa hoje, de forma intencional,
`provenance = "unknown"`, `live_verified = false`, `captured_at = null`, enquanto a
origem física da fonte não puder ser comprovada. Este notebook nunca representa
`unknown` como `live`.

**Segurança:** o token tem o formato `dobslit_qrng_live_<hex>` e é emitido na aba
*Desenvolvedor → Token* do portal. **Nunca** escreva um token real numa célula, num
arquivo versionado ou num print. Use uma variável de ambiente.

In [ ]:
import os, json, base64, hashlib, time, urllib.request, urllib.parse, urllib.error

BASE_URL        = "https://bongo.dobslit.com/qrng/v1"
PUBLIC_BASE_URL = "https://bongo.dobslit.com/qrng/v1/public"  # sem token, cota reduzida

# Placeholder — defina KAPUA_API_TOKEN no ambiente. NUNCA cole um token real aqui.
API_TOKEN  = os.environ.get("KAPUA_API_TOKEN", "SEU_TOKEN_AQUI")
HAVE_TOKEN = API_TOKEN not in ("", "SEU_TOKEN_AQUI")
TIMEOUT_S  = 30

print("token detectado:" , "sim" if HAVE_TOKEN else "não (usando endpoints públicos)")
# Evite imprimir o token. Se precisar confirmar qual token está carregado, mostre só o prefixo:
if HAVE_TOKEN:
    print("prefixo do token:", API_TOKEN[:18] + "…")

## Cliente HTTP mínimo (sem dependências)

Se preferir `requests`:
```python
import requests
H = {"Authorization": f"Bearer {API_TOKEN}"}
r = requests.get(f"{BASE_URL}/random", params={"bytes": 32, "format": "hex"}, headers=H, timeout=30)
r.raise_for_status(); print(r.json())
```
As células abaixo usam apenas a biblioteca padrão para rodarem em qualquer kernel.

In [ ]:
class KapuaError(RuntimeError):
    def __init__(self, status, error, message, request_id):
        super().__init__(f"[{status}] {error}: {message} (request_id={request_id})")
        self.status, self.error, self.message, self.request_id = status, error, message, request_id

def kapua_get(path, params=None, *, binary=False, public=False):
    """GET no Kapuã. Retorna (headers_dict, corpo). corpo = bytes se binary, senão dict."""
    use_public = public or not HAVE_TOKEN
    url = f"{PUBLIC_BASE_URL if use_public else BASE_URL}{path}"
    if params:
        url += "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, method="GET")
    if not use_public:
        req.add_header("Authorization", f"Bearer {API_TOKEN}")  # <-- autenticação
    req.add_header("Accept", "application/octet-stream" if binary else "application/json")
    try:
        with urllib.request.urlopen(req, timeout=TIMEOUT_S) as resp:
            headers = {k.lower(): v for k, v in resp.headers.items()}
            body = resp.read()
            return headers, (body if binary else json.loads(body.decode("utf-8")))
    except urllib.error.HTTPError as e:
        txt = e.read().decode("utf-8", "replace")
        try:
            j = json.loads(txt)
        except ValueError:
            j = {}
        raise KapuaError(e.code, j.get("error", "HTTP_ERROR"),
                         j.get("message", txt[:200]), j.get("request_id")) from None
    except urllib.error.URLError as e:
        raise KapuaError(0, "NETWORK", str(e.reason), None) from None

def show_provenance(headers, detail):
    d = detail or {}
    prov = d.get("actual_origin") or headers.get("x-qrng-provenance") or "unknown"
    live = d.get("live_verified")
    if live is None:
        live = headers.get("x-qrng-live-verified") == "true"
    print(f"  proveniência efetiva : {prov}")
    print(f"  live_verified        : {bool(live)}")
    print(f"  captured_at          : {d.get('captured_at', headers.get('x-qrng-captured-at'))!r}")
    print(f"  transport/buffer/entropy_health : "
          f"{d.get('transport_health', headers.get('x-qrng-transport-health'))} / "
          f"{d.get('buffer_health', headers.get('x-qrng-buffer-health'))} / "
          f"{d.get('entropy_health', headers.get('x-qrng-entropy-health'))}")
    if prov != "live":
        print("  NOTA: esta resposta NÃO é uma captura live verificada.")

## 1. Verificar a saúde (`GET /v1/health` — exige token)

In [ ]:
if HAVE_TOKEN:
    try:
        h, body = kapua_get("/health")
        print("status:", body.get("status"), "| api:", body.get("api"), "| request_id:", body.get("request_id"))
        up = body.get("upstream") or {}
        for k in ("source_status", "buffer_bytes_available", "stream_format", "sample_width_bytes", "conditioned"):
            if k in up:
                print(f"  upstream.{k} = {up[k]!r}")
    except KapuaError as e:
        print("health indisponível:", e)
else:
    print("pulado: /v1/health exige token. Defina KAPUA_API_TOKEN e reexecute.")

# Liveness do processo Node, SEM token e SEM consultar o upstream (rota /v1/health/self):
with urllib.request.urlopen(f"{BASE_URL}/health/self", timeout=TIMEOUT_S) as r:
    print("health/self:", json.loads(r.read().decode()))

## 2. Bytes brutos (`format=raw`) + salvar `.bin` + SHA-256 + request_id + proveniência

In [ ]:
N = 256
headers, raw = kapua_get("/random", {"bytes": N, "format": "raw"}, binary=True)
assert len(raw) == N, f"esperado {N} bytes, recebido {len(raw)}"          # N solicitados == N entregues
assert not raw[:3] == b"\xef\xbb\xbf", "não deve haver BOM"
print(f"recebidos {len(raw)} bytes | Content-Length={headers.get('content-length')} | "
      f"Content-Type={headers.get('content-type')}")
print("request_id:", headers.get("x-request-id"))
print("SHA-256   :", hashlib.sha256(raw).hexdigest())
show_provenance(headers, None)

with open("kapua_sample.bin", "wb") as f:
    f.write(raw)
print("salvo em kapua_sample.bin")

## 3. Hexadecimal — solicitar e decodificar

In [ ]:
headers, body = kapua_get("/random", {"bytes": N, "format": "hex"})
hex_str = body["random"]
assert len(hex_str) == 2 * body["bytes"], "hex tem 2 caracteres por byte"
assert set(hex_str) <= set("0123456789abcdef"), "hex é [0-9a-f]"
hex_bytes = bytes.fromhex(hex_str)                                        # decode(hex)
print("hex[:32] :", hex_str[:32])
print("bytes    :", len(hex_bytes), "| SHA-256:", hashlib.sha256(hex_bytes).hexdigest())
print("request_id:", body.get("request_id"))
show_provenance(headers, body.get("provenance_detail"))

## 4. Base64 — solicitar e decodificar

In [ ]:
headers, body = kapua_get("/random", {"bytes": N, "format": "base64"})
b64_bytes = base64.b64decode(body["random"])                              # decode(base64)
assert len(b64_bytes) == body["bytes"]
print("base64[:24]:", body["random"][:24])
print("bytes      :", len(b64_bytes), "| SHA-256:", hashlib.sha256(b64_bytes).hexdigest())

## 5. uint8 — array de inteiros 0..255 e verificação de quantidade

In [ ]:
headers, body = kapua_get("/random", {"bytes": N, "format": "uint8"})
arr = body["random"]
assert isinstance(arr, list) and len(arr) == body["bytes"] == N           # N bytes solicitados == N valores entregues
assert all(isinstance(v, int) and 0 <= v <= 255 for v in arr), "uint8 fora de 0..255"
u8_bytes = bytes(arr)                                                     # bytes(uint8)
print("uint8[:16]:", arr[:16])
print("bytes     :", len(u8_bytes), "| SHA-256:", hashlib.sha256(u8_bytes).hexdigest())

## 6. Equivalência dos formatos — sobre UMA amostra, não sobre 4 chamadas

As células 2–5 fizeram **chamadas independentes** → são **amostras diferentes** e os
SHA-256 acima **não coincidem** (isso é o esperado de uma fonte de aleatoriedade).
Para provar que `raw == decode(hex) == decode(base64) == bytes(uint8)` corretamente,
pegue **uma única amostra** e reserialize/decodifique localmente:

In [ ]:
headers, sample = kapua_get("/random", {"bytes": 128, "format": "raw"}, binary=True)  # amostra única
as_hex = sample.hex()
as_b64 = base64.b64encode(sample).decode()
as_u8  = list(sample)

assert bytes.fromhex(as_hex)   == sample, "raw == decode(hex)"
assert base64.b64decode(as_b64) == sample, "raw == decode(base64)"
assert bytes(as_u8)             == sample, "raw == bytes(uint8)"
sha = hashlib.sha256(sample).hexdigest()
assert (hashlib.sha256(bytes.fromhex(as_hex)).hexdigest()
        == hashlib.sha256(base64.b64decode(as_b64)).hexdigest()
        == hashlib.sha256(bytes(as_u8)).hexdigest() == sha), "SHA-256 idêntico em todos os formatos"
print("OK — os 4 formatos representam exatamente os mesmos", len(sample), "bytes")
print("SHA-256 comum:", sha)

## 7. Contrato binário e endianness (`uint32-LE`)

O transporte declara `uint32-le`. Ler 4 bytes como `uint32` **little-endian**:
`x = b0 + b1·2^8 + b2·2^16 + b3·2^24`. Big-endian dá outro número — não use.

In [ ]:
import struct
buf = sample  # a amostra única da célula 6
le = struct.unpack_from("<I", buf, 0)[0]            # little-endian (correto)
be = struct.unpack_from(">I", buf, 0)[0]            # big-endian (errado para este contrato)
manual = buf[0] | (buf[1] << 8) | (buf[2] << 16) | (buf[3] << 24)
assert le == manual
print(f"primeiros 4 bytes: {list(buf[:4])}")
print(f"uint32 LE = {le}   uint32 BE = {be}")

# Float uniforme em [0,1): x / 2^32  (nunca >= 1, pois x <= 2^32 - 1)
u = le / 2**32
assert 0.0 <= u < 1.0
print(f"u = x / 2^32 = {u:.12f}")

# Tamanho não múltiplo de 4: os bytes finais que não formam uma palavra completa
# ficam de fora do array de uint32 (o frontend usa a mesma regra: i + 3 < len).
n_words = len(buf) // 4
words = list(struct.unpack_from(f"<{n_words}I", buf, 0))
print(f"{len(buf)} bytes -> {n_words} uint32 (sobra {len(buf) - 4*n_words} byte(s))")

## 8. Inteiro uniforme sem viés de módulo (rejection sampling sobre uint32)

In [ ]:
def randint_uniform(lo, hi, *, over=8):
    """Inteiro uniforme em [lo, hi] (inclusive). Rejeita o topo do intervalo de
    uint32 que não é múltiplo de range — elimina o modulo bias."""
    rng = hi - lo + 1
    limit = (2**32 // rng) * rng
    need = max(rng, 1) and (16 + 4 * over)
    headers, buf = kapua_get("/random", {"bytes": need, "format": "raw"}, binary=True)
    for i in range(0, len(buf) - 3, 4):
        x = struct.unpack_from("<I", buf, i)[0]
        if x < limit:
            return lo + (x % rng)
    raise RuntimeError("rejection sampling não convergiu; peça mais bytes")

rolls = [randint_uniform(1, 6) for _ in range(10)]
print("10 lançamentos 1..6:", rolls)
assert all(1 <= r <= 6 for r in rolls)

## 9. Ler request ID e proveniência; consultar uso do token

In [ ]:
headers, body = kapua_get("/random", {"bytes": 16, "format": "hex"})
print("request_id (corpo) :", body.get("request_id"))
print("request_id (header):", headers.get("x-request-id"))
print("provenance (corpo) :", body.get("provenance"))
d = body.get("provenance_detail", {})
print("actual_origin      :", d.get("actual_origin"), "| live_verified:", d.get("live_verified"),
      "| captured_at:", d.get("captured_at"))

if HAVE_TOKEN:
    try:
        _, u = kapua_get("/me/usage")
        for k in ("quota_daily", "requests_today", "bytes_today"):
            if k in u:
                print(f"  {k} = {u[k]}")
    except KapuaError as e:
        print("  /me/usage:", e)

## 10. Tratamento de 401 / 403 / 429 / 503 e timeout

- **401 `MISSING_TOKEN`** — sem header `Authorization`.
- **403 `INVALID_TOKEN`** — header presente, token inválido/revogado.
- **429** — cota diária (`requests`/`bytes`) OU rate limit por IP (endpoint público) excedido.
- **503** — upstream indisponível/timeout ou entropia insuficiente no buffer (`INSUFFICIENT_ENTROPY`).

Regra: em 429/503, faça **backoff exponencial** e **não** reutilize bytes de uma resposta anterior.

In [ ]:
# 401 vs 403 (demonstração)
for label, hdr in (("sem header", None), ("token inválido", "Bearer nao-e-um-token")):
    req = urllib.request.Request(f"{BASE_URL}/random?bytes=8&format=hex")
    if hdr:
        req.add_header("Authorization", hdr)
    try:
        urllib.request.urlopen(req, timeout=TIMEOUT_S)
        print(label, "-> 200 (inesperado)")
    except urllib.error.HTTPError as e:
        print(f"{label:16} -> HTTP {e.code}: {e.read().decode()[:100]}")

def get_with_retry(path, params, *, tries=5, base_delay=1.0, binary=False):
    for attempt in range(tries):
        try:
            return kapua_get(path, params, binary=binary)
        except KapuaError as e:
            if e.status in (429, 502, 503) and attempt < tries - 1:
                delay = base_delay * (2 ** attempt)
                print(f"  {e.status} {e.error}: aguardando {delay:.0f}s (tentativa {attempt+1}/{tries})")
                time.sleep(delay)
                continue
            raise
    raise RuntimeError("esgotadas as tentativas")

hdrs, body = get_with_retry("/random", {"bytes": 8, "format": "hex"})
print("ok com retry:", body["random"])

# timeout curto proposital:
try:
    old = TIMEOUT_S
    req = urllib.request.Request(f"{PUBLIC_BASE_URL}/random?bytes=8&format=hex")
    urllib.request.urlopen(req, timeout=0.001)
except Exception as e:
    print("timeout capturado:", type(e).__name__)

## Resumo

- Os 4 formatos (`raw`, `hex`, `base64`, `uint8`) representam **exatamente os mesmos bytes** de uma dada amostra.
- `N` bytes solicitados = `N` bytes entregues; sem BOM; sem bytes extras.
- `uint32` é **little-endian**; `u = x / 2^32 ∈ [0, 1)`.
- O **token autentica e mede cota** — não transforma nem melhora os dados.
- A resposta informa `provenance`/`live_verified`/`captured_at`; hoje a produção
  reporta `unknown` / `false` / `null` de propósito — **não** trate como `live`.
- Geração criptográfica (chaves/seeds/nonces) está **indisponível** na API.
- Nunca imprima nem versione o token.